# Mounting Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [ ]:
%cd /content/drive/MyDrive/MSDS_422_Group/Final_Project

/content/drive/.shortcut-targets-by-id/1WwaHmrRiHi1g3fW23Dk9Awwek-1Dptvd/MSDS_422_Group/Final_Project


# Imports

In [ ]:
from random import seed
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split, ParameterGrid
from sklearn.preprocessing import LabelEncoder
from sklearn.utils.class_weight import compute_class_weight
from tensorflow.keras.applications import ResNet50, MobileNetV2, EfficientNetB0, InceptionV3
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from tensorflow.keras.layers import Dense, Dropout, GlobalAveragePooling2D
from tensorflow.keras.models import Model, load_model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.preprocessing import image
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.utils import array_to_img, img_to_array, load_img
import joblib
import json
import matplotlib.image as mpimg
import matplotlib.pyplot as plt
import nbconvert
import numpy as np
import os
import pandas as pd
import seaborn as sns
import tensorflow as tf

%matplotlib inline
sns.set()

In [ ]:
input_dir = os.path.join(os.getcwd(),'inputs_original')
output_dir = os.path.join(os.getcwd(),'outputs')

In [ ]:
print("Tensorflow version " + tf.__version__)

try:
  tpu = tf.distribute.cluster_resolver.TPUClusterResolver()  # TPU detection
  print(f'Running on a TPU w/{tpu.num_accelerators()["TPU"]} cores')
except ValueError:
  raise BaseException('ERROR: Not connected to a TPU runtime; please see the previous cell in this notebook for instructions!')

tf.config.experimental_connect_to_cluster(tpu)
tf.tpu.experimental.initialize_tpu_system(tpu)
tpu_strategy = tf.distribute.TPUStrategy(tpu)

Tensorflow version 2.15.0
Running on a TPU w/8 cores


#Data Ingestion

First only ingesting the annotations. The collection of images is large and does not make sense to load into memory at one time.

In [ ]:
# Opening JSON file that has the annotations for the training data set
f = open(os.path.join(input_dir,"train2019.json"))
# returns JSON object as a dictionary - assign to an object
training_annotations_dict = json.load(f)
# Closing file
f.close()

# Opening JSON file that has the annotations for the validation data set
f = open(os.path.join(input_dir,"val2019.json"))
# returns JSON object as a dictionary - assign to an object
val_annotations_dict = json.load(f)
# Closing file
f.close()

# Opening JSON file that has the unobfuscated category names
f = open(os.path.join(input_dir,"categories.json"))
# returns JSON object as a dictionary - assign to an object
clear_categories = json.load(f)
# Closing file
f.close()

# Opening JSON file that has the test info
f = open(os.path.join(input_dir,"test2019.json"))
# returns JSON object as a dictionary - assign to an object
test_annotations_dict = json.load(f)
# Closing file
f.close()

##Concise Generation of Training and Validation Subsets

In [ ]:
# create data frame from unobfuscated categories...
# 'clear_categories' was initially defined above from a separate .json file.
# here we redefine it as a data frame
rows = []
for c in range(0, len(clear_categories)):
  rows.append(clear_categories[c])

clear_categories = pd.DataFrame(rows)

###Training

In [ ]:
# pull annotations
train_df = pd.DataFrame(training_annotations_dict['annotations'])[['image_id','category_id']]

# pull image info (filenames)
train_img = pd.DataFrame(training_annotations_dict['images'])[['id', 'file_name']].rename(columns={'id':'image_id'})

# merge the two above into one data frame
train_df = train_df.merge(train_img, on='image_id')

# merge in class information
train_df = train_df.merge(clear_categories, left_on='category_id', right_on='id')

# convert id columns to string since they are nominal
train_df['category_id'] = train_df['category_id'].astype(str)
train_df['image_id'] = train_df['image_id'].astype(str)

# view result
# train_df.head()

###Validation

In [ ]:
# pull annotations
val_df = pd.DataFrame(val_annotations_dict['annotations'])[['image_id','category_id']]

# pull image info (filenames)
val_img = pd.DataFrame(val_annotations_dict['images'])[['id', 'file_name']].rename(columns={'id':'image_id'})

# merge the two above into one data frame
val_df = val_df.merge(val_img, on='image_id')

# merge in class information
val_df = val_df.merge(clear_categories, left_on='category_id', right_on='id')

# convert id columns to string since they are nominal
val_df['category_id'] = val_df['category_id'].astype(str)
val_df['image_id'] = val_df['image_id'].astype(str)

# view result
# val_df.head()

###Filtering to Just Amphibians

In [ ]:
amphibians_train_annotations = train_df[train_df['class'] == 'Amphibia']

amphibians_val_annotations = val_df[val_df['class'] == 'Amphibia']

###Rendering Streamlined Annotation Files

In [ ]:
amphibians_train_labels_simple = amphibians_train_annotations[['image_id', 'category_id', 'file_name']]

amphibians_test_labels_simple = amphibians_val_annotations[['image_id', 'category_id', 'file_name']]

##Data Pre-Processing and Feature Engineering

In [ ]:
# Verify file existence
def verify_filenames(df, directory, filename_col):
    valid_filenames = [f for f in df[filename_col] if os.path.isfile(os.path.join(directory, f))]
    return df[df[filename_col].isin(valid_filenames)]

# # Filter out invalid filenames from train_labels_simple
# train_labels_cleaned = verify_filenames(train_labels_simple, input_dir, 'file_name')
train_labels_cleaned = verify_filenames(amphibians_train_labels_simple, input_dir, 'file_name')

# # Filter out invalid filenames from val_df
# val_df_cleaned = verify_filenames(val_df, input_dir, 'file_name')
val_df_cleaned = verify_filenames(amphibians_test_labels_simple, input_dir, 'file_name')

# Check if the 'category_id' column exists and is valid
if 'category_id' not in train_labels_cleaned.columns or 'category_id' not in val_df_cleaned.columns:
    raise KeyError("The 'category_id' column is missing from the cleaned dataframes")

# Check the number of unique classes again
num_classes_train_cleaned = train_labels_cleaned['category_id'].nunique()
num_classes_val_cleaned = val_df_cleaned['category_id'].nunique()

print(f"Number of classes in cleaned training set: {num_classes_train_cleaned}")
print(f"Number of classes in cleaned validation set: {num_classes_val_cleaned}")

# Ensure both are the same
assert num_classes_train_cleaned == num_classes_val_cleaned, "Number of classes in cleaned train and validation sets do not match!"
num_classes = num_classes_train_cleaned  # Assuming they match

Number of classes in cleaned training set: 10
Number of classes in cleaned validation set: 10


In [ ]:
train_labels_cleaned_subset = train_labels_cleaned
val_df_cleaned_subset = val_df_cleaned

# Model Building

## Scratch Code

In [ ]:
# # Define paths for the train and test directories
# amphibians_dir = "inputs_original/train_val2019/Amphibians"

# # List all subdirectories (i.e., classes)
# classes = os.listdir(amphibians_dir)

# # Create a DataFrame to hold image paths and their corresponding labels
# image_paths = []
# labels = []

# for cls in classes:
#     class_dir = os.path.join(amphibians_dir, cls)
#     for img in os.listdir(class_dir):
#         image_paths.append(os.path.join(class_dir, img))
#         labels.append(cls)

# data = pd.DataFrame({
#     'image_path': image_paths,
#     'label': labels
# })

# # Split data into train (60%), validation (20%), and test (20%)
# train_data, temp_data = train_test_split(data, test_size=0.4, stratify=data['label'], random_state=42)
# val_data, test_data = train_test_split(temp_data, test_size=0.5, stratify=temp_data['label'], random_state=42)

# # Save the paths and labels
# train_images = train_data['image_path'].values
# train_labels = train_data['label'].values

# val_images = val_data['image_path'].values
# val_labels = val_data['label'].values

# test_images = test_data['image_path'].values
# test_labels = test_data['label'].values

### Class Weights for Imbalanced Classes

In [ ]:
# # Compute class weights
# class_weights = compute_class_weight(
#     class_weight='balanced',
#     classes=np.unique(train_generator.classes),
#     y=train_generator.classes
# )

# class_weights_dict = dict(enumerate(class_weights))

## ResNet50

Ideas:

 - Add early stopping rounds
 - Add dropout layers
 - Add regularization
 - Enhance data augmentation (to create more variations)
 - Change batch size (16, 32, etc.)

In [ ]:
seed(56)

### Basic Image Augmentation

In [ ]:
# Data Augmentation for training
train_datagen_1 = ImageDataGenerator(
    rescale=1.0/255,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True
)

# Only rescale for validation and test
val_test_datagen_1 = ImageDataGenerator(rescale=1.0/255)

# Generators
train_generator = train_datagen_1.flow_from_dataframe(
    dataframe=train_labels_cleaned_subset,
    directory=input_dir,
    x_col='file_name',
    y_col='category_id',
    target_size=(224, 224),
    batch_size=128,
    class_mode='categorical',
    shuffle=True
)

validation_generator = val_test_datagen_1.flow_from_dataframe(
    dataframe=val_df_cleaned_subset,
    directory=input_dir,
    x_col='file_name',
    y_col='category_id',
    target_size=(224, 224),
    batch_size=128,
    class_mode='categorical',
    shuffle=False
)

Found 3882 validated image filenames belonging to 10 classes.
Found 30 validated image filenames belonging to 10 classes.


#### Attempt 1

In [ ]:
def create_baseline_model():
  # Load the ResNet50 model with pre-trained ImageNet weights
  base_model = ResNet50(weights='imagenet', include_top=False, input_shape=(224, 224, 3))

  # Freeze the base model layers to prevent them from being trained
  for layer in base_model.layers:
      layer.trainable = False

  # Add custom layers on top of the ResNet50 base
  x = base_model.output
  x = GlobalAveragePooling2D()(x)
  x = Dense(1024, activation='relu')(x)
  predictions = Dense(len(train_generator.class_indices), activation='softmax')(x)

  # Define the model
  model = Model(inputs=base_model.input, outputs=predictions)

  # Compile the model
  model.compile(optimizer=Adam(learning_rate=0.001), loss='categorical_crossentropy', metrics=['accuracy'])

  return model

In [ ]:
with tpu_strategy.scope(): # creating the model in the TPUStrategy scope means we will train the model on the TPU
  ResNet50_v1_1 = create_baseline_model()

  history = ResNet50_v1_1.fit(
      train_generator,
      validation_data=validation_generator,
      epochs=10
  )

  # Evaluate on the validation set
  val_loss, val_acc = ResNet50_v1_1.evaluate(validation_generator)
  print(f"Validation Loss: {val_loss:.4f}, Validation Accuracy: {val_acc:.4f}")

Epoch 1/10
31/31 [==============================] - 1085s 35s/step - loss: 2.2388 - accuracy: 0.1574 - val_loss: 2.5877 - val_accuracy: 0.0667
Epoch 2/10
31/31 [==============================] - 71s 2s/step - loss: 2.1441 - accuracy: 0.1780 - val_loss: 2.5015 - val_accuracy: 0.1000
Epoch 3/10
31/31 [==============================] - 71s 2s/step - loss: 2.1401 - accuracy: 0.1865 - val_loss: 2.5286 - val_accuracy: 0.0333
Epoch 4/10
31/31 [==============================] - 71s 2s/step - loss: 2.1310 - accuracy: 0.1901 - val_loss: 2.6485 - val_accuracy: 0.0667
Epoch 5/10
31/31 [==============================] - 71s 2s/step - loss: 2.1414 - accuracy: 0.1968 - val_loss: 2.4931 - val_accuracy: 0.0667
Epoch 6/10
31/31 [==============================] - 72s 2s/step - loss: 2.0981 - accuracy: 0.2058 - val_loss: 2.5837 - val_accuracy: 0.1333
Epoch 7/10
31/31 [==============================] - 72s 2s/step - loss: 2.1037 - accuracy: 0.2159 - val_loss: 2.5232 - val_accuracy: 0.1000
Epoch 8/10
31/31 

In [ ]:
ResNet50_v1_1.save('ResNet50 Models/ResNet50_v1_1.keras')

#### Attempt 2
Increasing number of Epochs from 10 -> 20 and implementing early stopping rounds

In [ ]:
def create_baseline_model_v2():
  # Load the ResNet50 model with pre-trained ImageNet weights
  base_model = ResNet50(weights='imagenet', include_top=False, input_shape=(224, 224, 3))

  # Freeze the base model layers to prevent them from being trained
  for layer in base_model.layers:
      layer.trainable = False

  # Add custom layers on top of the ResNet50 base
  x = base_model.output
  x = GlobalAveragePooling2D()(x)
  x = Dense(1024, activation='relu')(x)
  predictions = Dense(len(train_generator.class_indices), activation='softmax')(x)

  # Define the model
  model = Model(inputs=base_model.input, outputs=predictions)

  # Compile the model
  model.compile(optimizer=Adam(learning_rate=0.001), loss='categorical_crossentropy', metrics=['accuracy'])

  return model

In [ ]:
with tpu_strategy.scope(): # creating the model in the TPUStrategy scope means we will train the model on the TPU
  ResNet50_v1_2 = create_baseline_model_v2()

  # Create EarlyStopping callback
  early_stopping = EarlyStopping(
      monitor='val_accuracy',  # Metric to monitor
      patience=3,          # Number of epochs with no improvement after which training will be stopped
      restore_best_weights=True  # Restores the model weights from the epoch with the best value of the monitored metric
  )

  history = ResNet50_v1_2.fit(
      train_generator,
      validation_data=validation_generator,
      epochs=20,
      callbacks=[early_stopping]
  )

  # Evaluate on the validation set
  val_loss, val_acc = ResNet50_v1_2.evaluate(validation_generator)
  print(f"Validation Loss: {val_loss:.4f}, Validation Accuracy: {val_acc:.4f}")

Epoch 1/20
31/31 [==============================] - 93s 3s/step - loss: 2.2497 - accuracy: 0.1473 - val_loss: 2.4984 - val_accuracy: 0.1000
Epoch 2/20
31/31 [==============================] - 73s 2s/step - loss: 2.1782 - accuracy: 0.1780 - val_loss: 2.6156 - val_accuracy: 0.1333
Epoch 3/20
31/31 [==============================] - 72s 2s/step - loss: 2.1493 - accuracy: 0.1795 - val_loss: 2.5354 - val_accuracy: 0.0333
Epoch 4/20
31/31 [==============================] - 71s 2s/step - loss: 2.1383 - accuracy: 0.1788 - val_loss: 2.5048 - val_accuracy: 0.1000
Epoch 5/20
1/1 [==============================] - 1s 1s/step - loss: 2.6151 - accuracy: 0.1333
Validation Loss: 2.6151, Validation Accuracy: 0.1333


In [ ]:
ResNet50_v1_2.save('ResNet50 Models/ResNet50_v1_2.keras')

### Enhanced Image Augmentation

#### Attempt 1

In [ ]:
train_datagen_2 = ImageDataGenerator(
    rescale=1.0/255,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,
    rotation_range=20,
    width_shift_range=0.2,
    height_shift_range=0.2,
    brightness_range=[0.8, 1.2],
    channel_shift_range=0.1,
    vertical_flip=True
)

# Only rescale for validation and test
val_test_datagen_2 = ImageDataGenerator(rescale=1.0/255)

# Generators
train_generator = train_datagen_2.flow_from_dataframe(
    dataframe=train_labels_cleaned_subset,
    directory=input_dir,
    x_col='file_name',
    y_col='category_id',
    target_size=(224, 224),
    batch_size=128,
    class_mode='categorical',
    shuffle=True
)

validation_generator = val_test_datagen_2.flow_from_dataframe(
    dataframe=val_df_cleaned_subset,
    directory=input_dir,
    x_col='file_name',
    y_col='category_id',
    target_size=(224, 224),
    batch_size=128,
    class_mode='categorical',
    shuffle=False
)

Found 3882 validated image filenames belonging to 10 classes.
Found 30 validated image filenames belonging to 10 classes.


In [ ]:
def create_enhanced_model_v1():
  # Load the ResNet50 model with pre-trained ImageNet weights
  base_model = ResNet50(weights='imagenet', include_top=False, input_shape=(224, 224, 3))

  # Freeze the base model layers to prevent them from being trained
  for layer in base_model.layers:
      layer.trainable = False

  # Add custom layers on top of the ResNet50 base
  x = base_model.output
  x = GlobalAveragePooling2D()(x)
  x = Dense(1024, activation='relu')(x)
  predictions = Dense(len(train_generator.class_indices), activation='softmax')(x)

  # Define the model
  model = Model(inputs=base_model.input, outputs=predictions)

  # Compile the model
  model.compile(optimizer=Adam(learning_rate=0.001), loss='categorical_crossentropy', metrics=['accuracy'])

  return model

In [ ]:
with tpu_strategy.scope(): # creating the model in the TPUStrategy scope means we will train the model on the TPU
  ResNet50_v2_1 = create_enhanced_model_v1()

  # Create EarlyStopping callback
  early_stopping = EarlyStopping(
      monitor='val_accuracy',  # Metric to monitor
      patience=3,          # Number of epochs with no improvement after which training will be stopped
      restore_best_weights=True  # Restores the model weights from the epoch with the best value of the monitored metric
  )

  history = ResNet50_v2_1.fit(
      train_generator,
      validation_data=validation_generator,
      epochs=20,
      callbacks=[early_stopping]
  )

  # Evaluate on the validation set
  val_loss, val_acc = ResNet50_v2_1.evaluate(validation_generator)
  print(f"Validation Loss: {val_loss:.4f}, Validation Accuracy: {val_acc:.4f}")

Epoch 1/20
31/31 [==============================] - 104s 3s/step - loss: 2.3333 - accuracy: 0.1368 - val_loss: 2.5254 - val_accuracy: 0.0667
Epoch 2/20
31/31 [==============================] - 81s 3s/step - loss: 2.1912 - accuracy: 0.1561 - val_loss: 2.5396 - val_accuracy: 0.0667
Epoch 3/20
31/31 [==============================] - 81s 3s/step - loss: 2.1793 - accuracy: 0.1468 - val_loss: 2.5099 - val_accuracy: 0.1333
Epoch 4/20
31/31 [==============================] - 81s 3s/step - loss: 2.1586 - accuracy: 0.1762 - val_loss: 2.5185 - val_accuracy: 0.0667
Epoch 5/20
31/31 [==============================] - 81s 3s/step - loss: 2.1564 - accuracy: 0.1713 - val_loss: 2.6149 - val_accuracy: 0.1333
Epoch 6/20
1/1 [==============================] - 1s 1s/step - loss: 2.5092 - accuracy: 0.1333
Validation Loss: 2.5092, Validation Accuracy: 0.1333


In [ ]:
ResNet50_v2_1.save('ResNet50 Models/ResNet50_v2_1.keras')

#### Attempt 2
Decreasing batch size from 128 to 16

In [ ]:
train_datagen_3 = ImageDataGenerator(
    rescale=1.0/255,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,
    rotation_range=20,
    width_shift_range=0.2,
    height_shift_range=0.2,
    brightness_range=[0.8, 1.2],
    channel_shift_range=0.1,
    vertical_flip=True
)

# Only rescale for validation and test
val_test_datagen_3 = ImageDataGenerator(rescale=1.0/255)

# Generators
train_generator = train_datagen_3.flow_from_dataframe(
    dataframe=train_labels_cleaned_subset,
    directory=input_dir,
    x_col='file_name',
    y_col='category_id',
    target_size=(224, 224),
    batch_size=16,
    class_mode='categorical',
    shuffle=True
)

validation_generator = val_test_datagen_3.flow_from_dataframe(
    dataframe=val_df_cleaned_subset,
    directory=input_dir,
    x_col='file_name',
    y_col='category_id',
    target_size=(224, 224),
    batch_size=16,
    class_mode='categorical',
    shuffle=False
)

Found 3882 validated image filenames belonging to 10 classes.
Found 30 validated image filenames belonging to 10 classes.


In [ ]:
with tpu_strategy.scope(): # creating the model in the TPUStrategy scope means we will train the model on the TPU
  ResNet50_v2_2 = create_enhanced_model_v1()

  # Create EarlyStopping callback
  early_stopping = EarlyStopping(
      monitor='val_accuracy',  # Metric to monitor
      patience=3,          # Number of epochs with no improvement after which training will be stopped
      restore_best_weights=True  # Restores the model weights from the epoch with the best value of the monitored metric
  )

  history = ResNet50_v2_2.fit(
      train_generator,
      validation_data=validation_generator,
      epochs=20,
      callbacks=[early_stopping]
  )

  # Evaluate on the validation set
  val_loss, val_acc = ResNet50_v2_2.evaluate(validation_generator)
  print(f"Validation Loss: {val_loss:.4f}, Validation Accuracy: {val_acc:.4f}")

Epoch 1/20
243/243 [==============================] - 107s 374ms/step - loss: 2.2822 - accuracy: 0.1422 - val_loss: 2.4156 - val_accuracy: 0.1000
Epoch 2/20
243/243 [==============================] - 80s 330ms/step - loss: 2.1567 - accuracy: 0.1654 - val_loss: 2.4855 - val_accuracy: 0.1000
Epoch 3/20
243/243 [==============================] - 80s 329ms/step - loss: 2.1496 - accuracy: 0.1669 - val_loss: 2.5091 - val_accuracy: 0.1000
Epoch 4/20
2/2 [==============================] - 1s 172ms/step - loss: 2.4152 - accuracy: 0.1000
Validation Loss: 2.4152, Validation Accuracy: 0.1000


In [ ]:
ResNet50_v2_2.save('ResNet50 Models/ResNet50_v2_2.keras')

### Grid Search Image Augmentation

In [ ]:
def datagen_gridsearch(params):
    train_datagen = ImageDataGenerator(
        rescale=1.0/255,
        shear_range=params['shear_range'],
        zoom_range=params['zoom_range'],
        horizontal_flip=params['horizontal_flip'],
        rotation_range=params['rotation_range'],
        width_shift_range=params['width_shift_range'],
        height_shift_range=params['height_shift_range'],
        brightness_range=params['brightness_range'],
        channel_shift_range=params['channel_shift_range'],
        vertical_flip=params['vertical_flip']
    )

    val_test_datagen = ImageDataGenerator(rescale=1.0/255)

    train_generator = train_datagen.flow_from_dataframe(
        dataframe=train_labels_cleaned_subset,
        directory=input_dir,
        x_col='file_name',
        y_col='category_id',
        target_size=(224, 224),
        batch_size=32, # can be altered
        class_mode='categorical',
        shuffle=True
    )

    validation_generator = val_test_datagen.flow_from_dataframe(
        dataframe=val_df_cleaned_subset,
        directory=input_dir,
        x_col='file_name',
        y_col='category_id',
        target_size=(224, 224),
        batch_size=32, # can be altered
        class_mode='categorical',
        shuffle=False
    )

    model = create_enhanced_model_v1()
    model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

    early_stopping = EarlyStopping(
        monitor='val_accuracy',
        patience=3,
        restore_best_weights=True
    )

    history = model.fit(
        train_generator,
        validation_data=validation_generator,
        epochs=20,
        callbacks=[early_stopping]
    )

    val_loss, val_acc = model.evaluate(validation_generator)
    return val_acc

In [ ]:
param_grid = {
    'shear_range': [0.1, 0.2],
    'zoom_range': [0.1, 0.2],
    'horizontal_flip': [True, False],
    'rotation_range': [10, 20],
    'width_shift_range': [0.1, 0.2],
    'height_shift_range': [0.1, 0.2],
    'brightness_range': [[0.8, 1.2], [0.7, 1.3]],
    'channel_shift_range': [0.1, 0.2],
    'vertical_flip': [True, False]
}

best_score = -np.inf
best_params = None

for params in ParameterGrid(param_grid):
    score = datagen_gridsearch(params)
    print(f"Params: {params}, Accuracy: {score:.4f}")
    if score > best_score:
        best_score = score
        best_params = params

print(f"Best Parameters: {best_params}")
print(f"Best Accuracy: {best_score:.4f}")

Found 3882 validated image filenames belonging to 10 classes.
Found 30 validated image filenames belonging to 10 classes.
Epoch 1/20
122/122 [==============================] - 114s 897ms/step - loss: 2.2762 - accuracy: 0.1610 - val_loss: 2.8013 - val_accuracy: 0.1000
Epoch 2/20
122/122 [==============================] - 109s 890ms/step - loss: 2.1794 - accuracy: 0.1615 - val_loss: 2.5202 - val_accuracy: 0.0667
Epoch 3/20
122/122 [==============================] - 109s 889ms/step - loss: 2.1495 - accuracy: 0.1636 - val_loss: 2.5473 - val_accuracy: 0.0667
Epoch 4/20
1/1 [==============================] - 1s 902ms/step - loss: 2.8013 - accuracy: 0.1000
Params: {'brightness_range': [0.8, 1.2], 'channel_shift_range': 0.1, 'height_shift_range': 0.1, 'horizontal_flip': True, 'rotation_range': 10, 'shear_range': 0.1, 'vertical_flip': True, 'width_shift_range': 0.1, 'zoom_range': 0.1}, Accuracy: 0.1000
Found 3882 validated image filenames belonging to 10 classes.
Found 30 validated image filena

# Evaluation

# Generate .html and then convert to PDF to include as appendix of report

In [ ]:
!jupyter nbconvert --to html /content/drive/MyDrive/MSDS_422_Group/Final_Project/iNaturalist_EDA_Data_Prep.ipynb

[NbConvertApp] WARNING | pattern '/content/drive/MyDrive/MSDS_422_Group/Final_Project/iNaturalist_EDA_Data_Prep.ipynb' matched no files
This application is used to convert notebook files (*.ipynb)
        to various other formats.


Options
The options below are convenience aliases to configurable class-options,
as listed in the "Equivalent to" description-line of the aliases.
To see all configurable class-options for some <cmd>, use:
    <cmd> --help-all

--debug
    set log level to logging.DEBUG (maximize logging output)
    Equivalent to: [--Application.log_level=10]
--show-config
    Show the application's configuration (human-readable format)
    Equivalent to: [--Application.show_config=True]
--show-config-json
    Show the application's configuration (json format)
    Equivalent to: [--Application.show_config_json=True]
--generate-config
    generate default config file
    Equivalent to: [--JupyterApp.generate_config=True]
-y
    Answer yes to any questions instead of promptin